<a href="https://colab.research.google.com/github/Essa-theresearcher/Essa-theresearcher/blob/main/siabates18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================
# DIABETIC RETINOPATHY (BINARY) - FULL COLAB PIPELINE
# Mount Drive -> Unzip -> Load Data -> Speed Optimizations -> Train -> Fine-tune -> Evaluate -> Save
# ============================

import os, shutil, numpy as np
import tensorflow as tf

# ---- (0) GPU sanity check ----
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if not gpus:
    print("\n[WARNING] No GPU detected. In Colab: Runtime -> Change runtime type -> GPU.\n")

# ---- (1) Mount Google Drive ----
from google.colab import drive
drive.mount("/content/drive")

# ---- (2) Paths (EDIT ONLY IF YOUR ZIP NAME/LOCATION DIFFERS) ----
ZIP_PATH = "/content/drive/MyDrive/diabetic-retinopathy-detection.zip"
EXTRACT_BASE = "/content/dr_dataset"
PROJECT_ROOT = os.path.join(EXTRACT_BASE, "diabetic-retinopathy-detection")

# The dataset folder inside your project (already confirmed):
DATA_ROOT = os.path.join(PROJECT_ROOT, "augmented_resized_binary")
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

# ---- (3) Unzip (fresh + overwrite, no prompts) ----
# If you re-run, this avoids conflicts and weird partial folders.
if os.path.exists(EXTRACT_BASE):
    shutil.rmtree(EXTRACT_BASE)
os.makedirs(EXTRACT_BASE, exist_ok=True)

print("\nUnzipping from Drive...")
!unzip -o -q "{ZIP_PATH}" -d "{EXTRACT_BASE}"

# Remove Mac metadata folder if present
mac_junk = os.path.join(EXTRACT_BASE, "__MACOSX")
if os.path.exists(mac_junk):
    shutil.rmtree(mac_junk)

print("\nProject root exists?", os.path.exists(PROJECT_ROOT))
print("Dataset root exists?", os.path.exists(DATA_ROOT))

# ---- (4) Confirm class folders ----
print("\nTrain classes:", os.listdir(TRAIN_DIR))
print("Val classes:", os.listdir(VAL_DIR))
print("Test classes:", os.listdir(TEST_DIR))

# ---- (5) Speed upgrades ----
# Mixed precision typically speeds up on T4/V100/A100 GPUs
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("\nMixed precision policy:", mixed_precision.global_policy())

# You can try 64 or 128 depending on GPU memory
BATCH = 64
IMG_SIZE = (224, 224)
SEED = 42

# ---- (6) Build datasets ----
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="binary",
    seed=SEED
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="binary",
    shuffle=False
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="binary",
    shuffle=False
)

class_names = train_ds_raw.class_names
print("Class names:", class_names)

AUTOTUNE = tf.data.AUTOTUNE

# cache() can increase RAM usage. With large datasets it may be too big.
# We keep prefetch always, and attempt cache cautiously.
train_ds = train_ds_raw.prefetch(AUTOTUNE)
val_ds   = val_ds_raw.prefetch(AUTOTUNE)
test_ds  = test_ds_raw.prefetch(AUTOTUNE)

# ---- (7) Build model (EfficientNetB0 transfer learning) ----
base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(224,224,3)
)
base.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)

# IMPORTANT: with mixed precision, keep final output float32 for numerical stability
outputs = tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

print("\nModel summary:")
model.summary()

# ---- (8) Callbacks ----
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=1),
    tf.keras.callbacks.ModelCheckpoint("dr_binary_phase1_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]

# ---- (9) Phase 1 training (frozen backbone) ----
EPOCHS_PHASE1 = 3
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks
)

# ---- (10) Phase 2 fine-tuning (unfreeze top layers) ----
# Unfreeze later layers, keep early layers frozen
base.trainable = True
fine_tune_at = int(len(base.layers) * 0.7)
for layer in base.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

callbacks_ft = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=1),
    tf.keras.callbacks.ModelCheckpoint("dr_binary_finetuned_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]

EPOCHS_FT = 3
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=callbacks_ft
)

# ---- (11) Evaluate on test ----
print("\nTest evaluation:")
test_metrics = model.evaluate(test_ds, verbose=0)
print(dict(zip(model.metrics_names, test_metrics)))

# ---- (12) Confusion matrix + classification report ----
# Install sklearn if needed (usually available in Colab)
try:
    from sklearn.metrics import confusion_matrix, classification_report
except Exception:
    !pip -q install scikit-learn
    from sklearn.metrics import confusion_matrix, classification_report

y_true = np.concatenate([y.numpy().astype(int) for _, y in test_ds], axis=0).ravel()
y_prob = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=class_names))

# If you want correct names, inspect class order:
print("\nClass order from dataset:", class_names)
# Note: image_dataset_from_directory maps class_names to integer labels in that order.

# ---- (13) Save to Google Drive ----
SAVE_DIR = "/content/drive/MyDrive/DR_Models"
os.makedirs(SAVE_DIR, exist_ok=True)

model_save_path = os.path.join(SAVE_DIR, "dr_binary_final.keras")
model.save(model_save_path)
print("\nSaved model to:", model_save_path)

# Save best checkpoints too
for f in ["dr_binary_phase1_best.keras", "dr_binary_finetuned_best.keras"]:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(SAVE_DIR, f))
        print("Copied checkpoint to Drive:", os.path.join(SAVE_DIR, f))

print("\nDONE.")


TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Unzipping from Drive...

Project root exists? True
Dataset root exists? True

Train classes: ['healthy', 'affected']
Val classes: ['healthy', 'affected']
Test classes: ['healthy', 'affected']

Mixed precision policy: <DTypePolicy "mixed_float16">
Found 115241 files belonging to 2 classes.
Found 14227 files belonging to 2 classes.
Found 14201 files belonging to 2 classes.
Class names: ['affected', 'healthy']
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

Model summary:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

Epoch 1/3
1801/1801 ━━━━━━━━━━━━━━━━━━━━ 416s 196ms/step - accuracy: 0.7216 - auc: 0.7950 - loss: 0.5365 - precision: 0.6958 - recall: 0.7377 - val_accuracy: 0.8022 - val_auc: 0.8809 - val_loss: 0.4306 - val_precision: 0.7899 - val_recall: 0.8064 - learning_rate: 0.0010
Epoch 2/3
1801/1801 ━━━━━━━━━━━━━━━━━━━━ 236s 131ms/step - accuracy: 0.7762 - auc: 0.8540 - loss: 0.4653 - precision: 0.7462 - recall: 0.8046 - val_accuracy: 0.8070 - val_auc: 0.8859 - val_loss: 0.4232 - val_precision: 0.7458 - val_recall: 0.9130 - learning_rate: 0.0010
Epoch 3/3
1801/1801 ━━━━━━━━━━━━━━━━━━━━ 269s 135ms/step - accuracy: 0.7807 - auc: 0.8572 - loss: 0.4598 - precision: 0.7505 - recall: 0.8091 - val_accuracy: 0.8147 - val_auc: 0.8886 - val_loss: 0.4129 - val_precision: 0.7638 - val_recall: 0.8941 - learning_rate: 0.0010
Epoch 1/3
1801/1801 ━━━━━━━━━━━━━━━━━━━━ 429s 196ms/step - accuracy: 0.7598 - auc: 0.8382 - loss: 0.5073 - precision: 0.7314 - recall: 0.7849 - val_accuracy: 0.8413 - val_auc: 0.9084 - va